In [21]:
import numpy as np

def accelerated_gradient_method(f_grad, x0, A, b, L0=1.0, eta=1.5,
                                max_iter=10000, tol=1e-8, restart=True):
    """
    Ускоренный градиентный метод
    params:
      f_grad: функция, возвращающая (f, grad) в точке x
      x0: начальная точка (должна удовлетворять Ax = b)
      A, b: матрица и вектор ограничений Ax = b (A полного строкового ранга)
      L0: начальная оценка константы Липшица градиента (по умолчанию 1.0)
      eta: множитель увеличения L в backtracking (по умолчанию 1.5)
      max_iter: максимальное число итераций (10000)
      tol: допуск на норму проекции градиента в касательном пространстве (1e-8)
      restart: использовать рестарты при нарушении убывания (True)

    returns:
      x: оптимальная точка
      f: значение f в x
      k: число выполненных итераций
      nu: None (заглушка, при необходимости)
    """
    m, n = A.shape
    assert np.allclose(A @ x0, b)
    AAT = A @ A.T
    def proj(z):
        y = np.linalg.solve(AAT, A @ z - b)
        return z - A.T @ y

    def proj_tangent(g):
        y = np.linalg.solve(AAT, A @ g)
        return g - A.T @ y

    x = x0.copy()
    y = x0.copy()
    t = 1.0
    L = L0

    for k in range(1, max_iter + 1):
        f_y, g_y = f_grad(y)
        if not np.isfinite(f_y):
            L = L0
            y = x.copy()
            t = 1.0
            continue

        L_try = L
        while True:
            x_new = proj(y - (1.0 / L_try) * g_y)
            f_new, _ = f_grad(x_new)
            if np.isfinite(f_new):
                diff = x_new - y
                Q = f_y + np.dot(g_y, diff) + 0.5 * L_try * np.dot(diff, diff)
                if f_new <= Q + 1e-12:
                    break
            L_try *= eta

        L = max(L_try / eta, 1e-12)

        x_prev = x.copy()
        x = x_new

        if restart and f_new > f_grad(x_prev)[0] + 1e-12:
            t = 1.0
            y = x.copy()
        else:
            t_prev = t
            t = 0.5 * (1.0 + np.sqrt(1.0 + 4.0 * t_prev * t_prev))
            y = x + ((t_prev - 1.0) / t) * (x - x_prev)

        _, g = f_grad(x)
        if np.linalg.norm(proj_tangent(g)) < tol:
            return x, -f_grad(x)[0], k, None
    return x, -f_grad(x)[0], max_iter, None

In [22]:
def test_accelerated_gradient():
    np.random.seed(0)
    n, m = 3, 5
    P = np.random.rand(m, n) + 0.5
    pi = np.random.dirichlet(np.ones(m))

    def f_grad(x):
        vals = P @ x
        if np.any(vals <= 0):
            return np.inf, np.zeros_like(x)
        f = -np.sum(pi * np.log(vals))
        grad = -P.T @ (pi / vals)
        return f, grad

    A = np.ones((1, n))
    b = np.array([1.0])
    x0 = np.ones(n) / n

    x_opt, f_opt, iters, _ = accelerated_gradient_method(
        f_grad, x0, A, b,
        L0=1.0, eta=1.5, max_iter=5000, tol=1e-8, restart=True
    )

    # Допустимость
    assert np.abs(np.sum(x_opt) - 1.0) < 1e-10, "Сумма ≠ 1"
    assert np.all(P @ x_opt > 0), "p_j^T x <= 0"

    # Стационарность
    _, grad = f_grad(x_opt)
    proj = grad - np.mean(grad)
    norm_proj = np.linalg.norm(proj)
    assert norm_proj < 1e-6, f"|proj(∇f)| = {norm_proj:.2e}"

    # Сравнение с CVXPY
    try:
        import cvxpy as cp
        x_var = cp.Variable(n)
        constraints = [cp.sum(x_var) == 1]
        objective = cp.Maximize(cp.sum(pi @ cp.log(P @ x_var)))
        prob = cp.Problem(objective, constraints)
        prob.solve(solver=cp.SCS, verbose=False, eps=1e-8)
        f_cvxpy = prob.value
        print(f"FISTA f_opt = {f_opt:.6f}, CVXPY f_opt = {f_cvxpy:.6f}, разница = {abs(f_opt - f_cvxpy):.2e}")
        assert abs(f_opt - f_cvxpy) < 1e-4, f"Расхождение с CVXPY > 1e-4"
        print(f"Тест пройден с CVXPY! Итераций: {iters}")
    except ImportError:
        print(f"Тест пройден: f_opt = {f_opt:.6f}, |proj(∇f)| = {norm_proj:.2e}, итераций = {iters}")

test_accelerated_gradient()

FISTA f_opt = 1.769492, CVXPY f_opt = 1.769492, разница = 4.71e-11
Тест пройден с CVXPY! Итераций: 5000
